In [ ]:
%load_ext autoreload
%autoreload 2

# pywatemsedem Postprocessing

## 1. Introduction

This tutorial demonstrates how to use the **PostProcess** class from the pywatemsedem package to analyze and postprocess WaTEM/SEDEM model output data.

What you will do in this notebook:
- Initialize a `PostProcess` object for a scenario.
- Inspect routing outputs (routing vectors, optional missing-routing vectors, and routing tables without river pixels).
- Explore sink-related vectors: sediment export to rivers, sewer inflow, and combined sinks.
- Inspect ranked sediment-load outputs for prioritization.
- Delineate and inspect point-of-interest (POI) subcatchments.
- Delineate and inspect buffer subcatchments.
- Identify and visualize priority subcatchments with two approaches:
  - top-`n` selection,
  - percentage-based selection using `sedi_export + sewer_in`.
- Run and inspect the grass-strip postprocessing workflow.

All generated outputs are written to the postprocessing folder and can be inspected as GeoDataFrames or loaded in GIS.

## 2. Imports and example data

In [ ]:
import os
from pathlib import Path

import pandas as pd

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

In [ ]:
# Set the working directory to the project root
os.chdir("../..")

# Select the example results folder used in this tutorial
home_folder = Path("c:/git/pywatemsedem/tests/postprocess/langegracht")

## 3. PostProcess object

### 3.1. Initialisation

The object initialisation arguments can be fetched based on ini file and the epsg code

In [ ]:
home_folder.exists()

In [ ]:
# Create the main PostProcess object used in all next steps
from pywatemsedem.postprocess import PostProcess
pp = PostProcess(home_folder, 20, 1, 2019, epsg=31370)

#### Routing Vector

In [ ]:
# Load the routing lines so you can inspect sediment pathways
pp.vct_routing

In [ ]:
# Check where the routing vector is saved on disk
pp.vct_routing.file_path

In [ ]:
# Quick preview of routing attributes and geometry
pp.vct_routing.geodata.head()

In [ ]:
pp.vct_routing.plot(show_mask=True, show_river=True, show_labels=False)

#### Missing Routing Vector

In [ ]:
# Optional: create a layer with routes that do not end in a sink
pp.make_routing_missing_vct()

In [ ]:
# Optional: check where the missing-routing layer is saved
# pp.vct_routing_missing.file_path

#### Routing table excluding river pixels

In [ ]:
# Inspect non-river routing links (land-to-land flow)
df_routing_non_river = pp.routing_non_river
df_routing_non_river

#### Sediment Export to Rivers vector

In [ ]:
# Load river export points to inspect where sediment reaches rivers
pp.vct_sedi_export

In [ ]:
# Check where the river export layer is saved
pp.vct_sedi_export.file_path

In [ ]:
# Quick preview of river export point attributes
pp.vct_sedi_export.geodata.head()

In [ ]:
pp.vct_sedi_export.plot(show_mask=True, show_river=True, show_labels=False)

#### Sewer Inflow vector

In [ ]:
# Load sewer inflow points to inspect where sediment enters sewers
pp.vct_sewer_in

In [ ]:
# Check where the sewer inflow layer is saved
pp.vct_sewer_in.file_path

In [ ]:
# Quick preview of sewer inflow point attributes
pp.vct_sewer_in.geodata.head()

In [ ]:
pp.vct_sewer_in.plot(show_mask=True, show_river=True, show_labels=False)

#### Combined Sinks (Rivers + Sewers)

In [ ]:
# Load the combined sinks layer (rivers + sewers) for total sink analysis
pp.vct_sinks

In [ ]:
# Check where the combined sinks layer is saved
pp.vct_sinks.file_path

In [ ]:
# Quick preview of combined sink attributes
pp.vct_sinks.geodata.head()

In [ ]:
pp.vct_sinks.plot(show_mask=True, show_river=True, show_labels=False)

#### Point-of-interest subcatchments (2 POIs)
We demonstrate the point workflow with exactly two POIs. The model delineates one subcatchment per point and then aggregates them on the parent points vector (`vct_poi.vct_subcatchments`).

In [ ]:
# Add two demonstration POIs for subcatchment delineation
_ = pp.add_poi(
    [165570.4, 164464.4],
    [168768, 166967.9],
    poi_id=[101, 102],
    filename="poi_demo_2.shp",
)

# Quick check: do the POIs look correct?
pp.vct_poi.geodata.head()

In [ ]:
pp.vct_poi.plot(show_mask=True, show_river=True)

In [ ]:
pp.vct_poi.file_path

In [ ]:
# Delineate subcatchments that drain to the POIs
pp.identify_subcatchments(
    target_input="vct_poi",
    id_column="poi_id",
    tag="subcatchments_to_poi",
)

# Quick check: inspect resulting subcatchments
pp.vct_poi.vct_subcatchments.geodata.head()

In [ ]:
pp.vct_poi.vct_subcatchments.file_path

Plot options

Use the default call:

```python
pp.vct_poi.vct_subcatchments.plot()
```

Most used options:

- `show_river=True/False`
- `show_labels=True/False`
- `column="VALUE"` (or `None`)
- `river_color`, `poi_color`, `alpha`

In [ ]:
pp.vct_poi.vct_subcatchments.plot(
    show_river=True,
    show_labels=True,
    alpha=0.5,
)

#### Buffer subcatchments

Use the same pattern as POI: first create `vct_buffers`, then delineate and access subcatchments via `vct_buffers.vct_subcatchments`.

In [ ]:
# Load buffers that will be used as target zones
pp.vct_buffers

In [ ]:
pp.vct_buffers.geodata.head()

In [ ]:
pp.vct_buffers.plot(show_mask=True, show_river=True)

In [ ]:
# Delineate subcatchments that drain to buffers
_ = pp.identify_subcatchments_to_buffers()

In [ ]:
# Run once more to make sure the latest plotting helper is attached
_ = pp.identify_subcatchments_to_buffers()

pp.vct_buffers.vct_subcatchments.plot(
    show_buffers=True,
    show_labels=True,
    show_river=True,
)

#### Priority subcatchments
In this section, we identify the most important subcatchments that contribute the most sediment.

What happens step by step:
1. We first compute the priority points.
2. Then we link each point to its corresponding subcatchment.
3. We always show `vct_priority_points` first, then `vct_priority_points.vct_subcatchments`.
4. The plot gives a quick visual overview of where the key areas are.

How to read the colors in the plot:
- red = highest priority
- blue = lower priority

Below, we show two methods with the same structure so they are easy to compare.

In [ ]:
# Top-n method: keep the 5 highest-priority locations
pp.identify_priority_subcatchments(source="sedi_out", approach="n", nmax=5)

In [ ]:
pp.vct_priority_points.plot(show_mask=True, show_river=True)

In [ ]:
# Check where the priority-points layer is saved
pp.vct_priority_points.file_path

In [ ]:
# Check where the priority-subcatchments layer is saved
pp.vct_priority_points.vct_subcatchments.file_path

In [ ]:
# Quick preview of priority-point attributes
pp.vct_priority_points.geodata.head()

In [ ]:
# Quick preview of priority-subcatchment attributes
pp.vct_priority_points.vct_subcatchments.geodata.head()

In [ ]:
pp.vct_priority_points.vct_subcatchments.plot(
    column="VALUE",
    cmap="RdBu",
    show_river=True,
    show_labels=True,
    alpha=0.5,
)

##### Priority subcatchments (approach 2: percentage, source = sedi_export + sewer_in)
This method keeps priority subcatchments until cumulative contribution first exceeds the chosen threshold.

In practice:
- `source="sedi_export + sewer_in"`: combines contribution to rivers and sewers.
- `approach="percentage"`: keeps subcatchments in priority order up to the first threshold crossing.
- `threshold=30`: keeps the highest-priority zones until `cumperc` first goes above 30%.

In the geodata for this method, you also get `cumperc`:
- `cumperc` is the cumulative contribution by priority level.
- It helps you verify where the first threshold crossing happens.

In [ ]:
# Percentage method: keep subcatchments until cumulative contribution first exceeds 30%
pp.identify_priority_subcatchments(
    source="sedi_export + sewer_in",
    approach="percentage",
    threshold=30,
    flag_merge=True,
)

In [ ]:
pp.vct_priority_points.plot(show_mask=True, show_river=True)

In [ ]:
# Check where the priority-points layer is saved
pp.vct_priority_points.file_path

In [ ]:
# Check where the priority-subcatchments layer is saved
pp.vct_priority_points.vct_subcatchments.file_path

In [ ]:
# Quick preview of priority-point attributes
pp.vct_priority_points.geodata.head()

In [ ]:
# Quick preview of priority-subcatchment attributes
pp.vct_priority_points.vct_subcatchments.geodata.head()

In [ ]:
pp.vct_priority_points.vct_subcatchments.plot(
    column="VALUE",
    cmap="RdBu",
    show_river=True,
    show_labels=True,
    alpha=0.5,
)

#### Grass strips

In [ ]:
# Provide the grass-strip layer used for the efficiency check
pp.vct_grass_strips = Path("tests/data/grass_strips.shp")

# Run grass-strip postprocessing
gdf = pp.process_grass_strips()

In [ ]:
gdf.loc[gdf["value"] > 0].head()

In [ ]:
pp.vct_grass_strips.plot(show_mask=True, show_river=True, cmap="Greens", column="eSTE")